<a href="https://colab.research.google.com/github/RatnaNalajala/python-sdk/blob/main/_downloads/6b6889455ef3d6c74e64c3fc1c12815b/dynamic_net.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# For tips on running notebooks in Google Colab, see
# https://docs.pytorch.org/tutorials/beginner/colab
%matplotlib inline

PyTorch: Control Flow + Weight Sharing
======================================

To showcase the power of PyTorch dynamic graphs, we will implement a
very strange model: a third-fifth order polynomial that on each forward
pass chooses a random number between 4 and 5 and uses that many orders,
reusing the same weights multiple times to compute the fourth and fifth
order.


In [3]:
import random
import torch
import math


class DynamicNet(torch.nn.Module):
    def __init__(self):
        """
        In the constructor we instantiate five parameters and assign them as members.
        """
        super().__init__()
        self.a = torch.nn.Parameter(torch.randn(()))
        self.b = torch.nn.Parameter(torch.randn(()))
        self.c = torch.nn.Parameter(torch.randn(()))
        self.d = torch.nn.Parameter(torch.randn(()))
        self.e = torch.nn.Parameter(torch.randn(()))

    def forward(self, x):
        """
        For the forward pass of the model, we randomly choose either 4, 5
        and reuse the e parameter to compute the contribution of these orders.

        Since each forward pass builds a dynamic computation graph, we can use normal
        Python control-flow operators like loops or conditional statements when
        defining the forward pass of the model.

        Here we also see that it is perfectly safe to reuse the same parameter many
        times when defining a computational graph.
        """
        y = self.a + self.b * x + self.c * x ** 2 + self.d * x ** 3
        for exp in range(4, random.randint(4, 6)):
            y = y + self.e * x ** exp
        return y

    def string(self):
        """
        Just like any class in Python, you can also define custom method on PyTorch modules
        """
        return f'y = {self.a.item()} + {self.b.item()} x + {self.c.item()} x^2 + {self.d.item()} x^3 + {self.e.item()} x^4 ? + {self.e.item()} x^5 ?'


# Create Tensors to hold input and outputs.
x = torch.linspace(-math.pi, math.pi, 2000)
y = torch.sin(x)

# Construct our model by instantiating the class defined above
model = DynamicNet()

# Construct our loss function and an Optimizer. Training this strange model with
# vanilla stochastic gradient descent is tough, so we use momentum
criterion = torch.nn.MSELoss(reduction='sum')
optimizer = torch.optim.SGD(model.parameters(), lr=1e-8, momentum=0.9)
for t in range(30000):
    # Forward pass: Compute predicted y by passing x to the model
    y_pred = model(x)

    # Compute and print loss
    loss = criterion(y_pred, y)
    if t % 2000 == 1999:
        print(t, loss.item())

    # Zero gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

print(f'Result: {model.string()}')

1999 524.8489990234375
3999 266.0980529785156
5999 124.19380950927734
7999 60.04802703857422
9999 32.78219985961914
11999 20.203737258911133
13999 14.063241958618164
15999 11.264593124389648
17999 9.93269157409668
19999 9.2313871383667
21999 8.912581443786621
23999 8.962254524230957
25999 8.871123313903809
27999 8.665702819824219
29999 8.88127326965332
Result: y = -0.002432913053780794 + 0.8553985953330994 x + -0.00014658589498139918 x^2 + -0.0935574397444725 x^3 + 0.00010749603097792715 x^4 ? + 0.00010749603097792715 x^5 ?
